In [1]:
import re, os, json, glob, math
from pathlib import Path
import numpy as np
import pandas as pd

# ---- CONFIG (edit these in Colab as needed) ----
LOG_DIR = "src/eval/logs/upstepper_transfer/"   # folder containing your run_ppo_*.log files
PATTERN = "run_ppo_*_run*.log"  # glob pattern
LAST_K_EVALS = 10      # mean of last K evaluation means within each run
TARGET_ENV = None      # optional: e.g. "Walker-v0" to filter eval lines by env tag (recommended for safety)
EVAL_INTERVAL_FALLBACK = 10000  # only used if log doesn't expose timesteps cleanly

# -----------------------------------------------

EVAL_LINE_RE = re.compile(
    r"\[([^\]]+)\]\s*Mean:\s*([0-9.+-eE]+),\s*Std:\s*([0-9.+-eE]+),\s*Min:\s*([0-9.+-eE]+),\s*Max:\s*([0-9.+-eE]+)"
)
TIMESTEP_RE = re.compile(r"\|\s*total_timesteps\s*\|\s*([0-9]+)\s*\|")
FNAME_RE = re.compile(
    r"run_ppo_(?P<A>.+?)_to_(?P<B>.+?)_t(?P<T>[0-9]+)_run(?P<R>[0-9]+)\.log$"
)

def parse_one_log(path: str, target_env: str | None = None):
    p = Path(path)
    m = FNAME_RE.search(p.name)
    if not m:
        raise ValueError(f"Filename doesn't match expected pattern: {p.name}")

    A = m.group("A")
    B = m.group("B")
    T = int(m.group("T"))
    run = int(m.group("R"))

    with open(p, "rb") as f:
        text = f.read().decode("utf-8", errors="ignore")

    # Extract eval lines
    eval_matches = EVAL_LINE_RE.findall(text)
    if not eval_matches:
        raise ValueError(f"No evaluation lines found in {p.name}")

    env_tags = [em[0] for em in eval_matches]
    means = [float(em[1]) for em in eval_matches]
    stds  = [float(em[2]) for em in eval_matches]
    mins  = [float(em[3]) for em in eval_matches]
    maxs  = [float(em[4]) for em in eval_matches]

    # Optional filter by target env tag
    if target_env is None:
        target_env = B  # sensible default: use B env from filename
    keep_idx = [i for i, tag in enumerate(env_tags) if tag == target_env]
    if keep_idx:
        env_tags = [env_tags[i] for i in keep_idx]
        means = [means[i] for i in keep_idx]
        stds  = [stds[i]  for i in keep_idx]
        mins  = [mins[i]  for i in keep_idx]
        maxs  = [maxs[i]  for i in keep_idx]

    # Extract timesteps (best-effort)
    ts_matches = [int(x) for x in TIMESTEP_RE.findall(text)]
    # These appear many times (every SB3 table). We want something aligned with evals.
    # Commonly, each eval line corresponds to the most recent total_timesteps printed before it.
    # We'll do a single pass through lines to align them.
    timesteps_aligned = []
    cur_ts = None
    means_aligned = []
    stds_aligned = []
    mins_aligned = []
    maxs_aligned = []
    env_aligned = []

    # Build a fast map of eval lines we care about (by exact substring match)
    # Instead of substring matching (fragile), we re-scan line-by-line and parse on the fly.
    for line in text.splitlines():
        mt = TIMESTEP_RE.search(line)
        if mt:
            cur_ts = int(mt.group(1))
        me = EVAL_LINE_RE.search(line)
        if me:
            tag = me.group(1)
            if tag != target_env:
                continue
            env_aligned.append(tag)
            means_aligned.append(float(me.group(2)))
            stds_aligned.append(float(me.group(3)))
            mins_aligned.append(float(me.group(4)))
            maxs_aligned.append(float(me.group(5)))
            timesteps_aligned.append(cur_ts)

    # Fallback: if we failed to align timesteps, synthesize by eval interval
    if not timesteps_aligned or all(ts is None for ts in timesteps_aligned):
        timesteps_aligned = [(i + 1) * EVAL_INTERVAL_FALLBACK for i in range(len(means))]
        means_aligned = means
        stds_aligned = stds
        mins_aligned = mins
        maxs_aligned = maxs
        env_aligned = [target_env] * len(means)

    df = pd.DataFrame({
        "timesteps": timesteps_aligned,
        "env": env_aligned,
        "mean": means_aligned,
        "std": stds_aligned,
        "min": mins_aligned,
        "max": maxs_aligned,
    }).dropna(subset=["mean"])

    # Ensure ordering
    df = df.sort_values("timesteps").reset_index(drop=True)

    # Compute final metric
    tail = df["mean"].tail(LAST_K_EVALS).to_numpy()
    final_score = float(np.mean(tail)) if len(tail) else float("nan")

    return {
        "file": str(p),
        "A": A,
        "B": B,
        "T": T,
        "run": run,
        "target_env": target_env,
        "n_evals": int(len(df)),
        "final_score_mean_lastK": final_score,
        "best_eval_mean": float(df["mean"].max()),
        "last_timestep_seen": int(df["timesteps"].max()) if len(df) else None,
        "df": df,
    }

# ---- Find and parse all matching logs ----
paths = sorted(glob.glob(os.path.join(LOG_DIR, PATTERN)))
if not paths:
    raise FileNotFoundError(f"No logs found in {LOG_DIR} matching {PATTERN}")

parsed = []
for path in paths:
    try:
        parsed.append(parse_one_log(path, target_env=TARGET_ENV))
    except Exception as e:
        print(f"Skipping {Path(path).name}: {e}")

if not parsed:
    raise RuntimeError("No logs successfully parsed. Check pattern and filename format.")

# Group by (A,B,T)
rows = []
for key, group in pd.DataFrame([{
        "A": p["A"], "B": p["B"], "T": p["T"], "run": p["run"],
        "final": p["final_score_mean_lastK"], "best": p["best_eval_mean"],
        "n_evals": p["n_evals"], "file": p["file"],
        "last_ts": p["last_timestep_seen"]
    } for p in parsed]).groupby(["A", "B", "T"]):

    g = group.sort_values("run")
    finals = g["final"].to_numpy(dtype=float)
    mean_final = float(np.mean(finals))
    std_final = float(np.std(finals, ddof=1)) if len(finals) > 1 else float("nan")

    rows.append({
        "A": key[0],
        "B": key[1],
        "total_timesteps": int(key[2]),
        "n_runs": int(len(g)),
        f"cell_value_mean_last{LAST_K_EVALS}": mean_final,
        "across_run_std": std_final,
        "run_finals": [float(x) for x in finals],
        "run_files": g["file"].tolist(),
        "run_best_eval_means": [float(x) for x in g["best"].to_numpy(dtype=float)],
        "min_evals_in_runs": int(g["n_evals"].min()),
        "max_evals_in_runs": int(g["n_evals"].max()),
        "max_timestep_in_runs": int(g["last_ts"].max()),
    })

summary_df = pd.DataFrame(rows).sort_values(["A","B","total_timesteps"]).reset_index(drop=True)
summary_df

FileNotFoundError: No logs found in src/eval/logs/upstepper_transfer/ matching run_ppo_*_run*.log